In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
import sys
print(sys.executable)

In [ ]:
# IMPORTS
# ================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryFocalCrossentropy

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# ================================
# PATHS + CONFIG
# ================================

train_path = '/Users/Ahmad/UNI-work/year3/FYP/Project/tuberculosis/datasets/cnn/train'
test_path = '/Users/Ahmad/UNI-work/year3/FYP/Project/tuberculosis/datasets/cnn/test'
valid_path = '/Users/Ahmad/UNI-work/year3/FYP/Project/tuberculosis/datasets/cnn/val'

img_height = 224
img_width = 224
batch_size = 16

In [ ]:
# ================================
# DATA GENERATORS
# ================================
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.08,
    height_shift_range=0.08,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

test_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(
    train_path,
    target_size=(img_height, img_width),
    color_mode='grayscale',
    class_mode='binary',
    batch_size=batch_size
)

valid = test_gen.flow_from_directory(
    valid_path,
    target_size=(img_height, img_width),
    color_mode='grayscale',
    class_mode='binary',
    batch_size=batch_size
)

test = test_gen.flow_from_directory(
    test_path,
    target_size=(img_height, img_width),
    color_mode='grayscale',
    class_mode='binary',
    shuffle=False,
    batch_size=batch_size
)

In [ ]:
# ================================
# CLASS WEIGHTS
# ================================
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train.classes),
    y=train.classes
)

class_weights = dict(zip(np.unique(train.classes), weights))
print("Class Weights:", class_weights)

In [ ]:
# ================================
# MODEL (IMPROVED CNN)
# ================================
cnn = Sequential()

cnn.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(2,2))

cnn.add(Conv2D(64, (3,3), activation='relu'))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(2,2))

cnn.add(Conv2D(128, (3,3), activation='relu'))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(2,2))

cnn.add(Conv2D(256, (3,3), activation='relu'))
cnn.add(BatchNormalization())
cnn.add(MaxPooling2D(2,2))

cnn.add(Flatten())

cnn.add(Dense(128, activation='relu'))
cnn.add(Dropout(0.5))

cnn.add(Dense(1, activation='sigmoid'))

cnn.compile(
    optimizer='adam',
    loss=BinaryFocalCrossentropy(gamma=2),
    metrics=['accuracy']
)

cnn.summary()

In [ ]:
# ================================
# CALLBACKS
# ================================
early = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, min_lr=1e-6)


In [ ]:
# ================================
# TRAINING
# ================================
history = cnn.fit(
    train,
    epochs=25,
    validation_data=valid,
    class_weight=class_weights,
    callbacks=[early, lr_reduce]
)


In [ ]:
# ================================
# EVALUATION
# ================================
test_loss, test_acc = cnn.evaluate(test)
print("\nTest Accuracy:", test_acc)


In [ ]:
# ================================
# PREDICTIONS
# ================================
preds = cnn.predict(test)

print("\nPrediction stats:")
print("Min:", preds.min())
print("Max:", preds.max())
print("Mean:", preds.mean())

In [ ]:
# ================================
# THRESHOLD TUNING
# ================================
best_f1 = 0
best_threshold = 0

for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    predictions = (preds > t).astype(int)
    report = classification_report(test.classes, predictions, output_dict=True)

    tb_f1 = report['1']['f1-score']

    print(f"\nThreshold: {t}")
    print(classification_report(test.classes, predictions, target_names=['Normal','TB']))

    if tb_f1 > best_f1:
        best_f1 = tb_f1
        best_threshold = t
        best_preds = predictions

print("\nBest Threshold:", best_threshold)
print("Best TB F1:", best_f1)


In [ ]:
# ================================
# CONFUSION MATRIX (BEST)
# ================================
cm = confusion_matrix(test.classes, best_preds)

plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','TB'],
            yticklabels=['Normal','TB'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix (Threshold={best_threshold})")
plt.show()